In [66]:
from IPython.display import display, Markdown

display(Markdown(r'''
$$
\begin{aligned}

\textbf{Return on carry trade:} \quad 
& G_S = S_T / S_0   \\[2pt]
X = G_S · Y_T / Y_0 - e^{r_{B_0}T} \\[10pt]

\textbf{Product payoff structure:} \quad 
& \Phi(r_{S_T}, Y_T) =
\begin{cases}
0, 
& \text{if } X > 0 \\

k(-0.5)X, 
& \text{if } -5\% < X < 0 \\

-0.5X, 
& \text{if } X < -5\%
\end{cases}
\\[6pt]
& \text{where } X = r_{S_T} \frac{Y_T}{Y_0} - e^{r_{B_0}T}, \quad
k = \frac{X}{-5\%} \text{ (proportion of loss compensated)}
\\[12pt]

\textbf{Interest rate processes:} \quad 
& \begin{cases}
\text{US: } dr_{A_t} = \kappa_A (\theta_A - r_{A_t})\,dt + \sigma_A dW_t^A \\
\text{Japan: } dr_{B_t} = \kappa_B (\theta_B - r_{B_t})\,dt + \sigma_B dW_t^B
\end{cases} \\[12pt]
                 
\textbf{Foreign exchange rate SDE:} \quad 
& dY_t = (r_{B_t} - r_{A_t}) Y_t\,dt + \sigma_Y Y_t\,dW_t^Y \\[10pt]

\textbf{Stock price process:} \quad 
& dS_t = r_{A_t} S_t\,dt + \sigma_S S_t\,dW_t^S

\end{aligned}
$$

$$
A = \text{USD}, \quad B = \text{JPY}, \quad Y_0 = \text{USD/JPY at } t=0, \quad Y_T = \text{USD/JPY at } t=T
$$
'''))


$$
\begin{aligned}

\textbf{Return on carry trade:} \quad 
& G_S = S_T / S_0   \\[2pt]
X = G_S · Y_T / Y_0 - e^{r_{B_0}T} \\[10pt]

\textbf{Product payoff structure:} \quad 
& \Phi(r_{S_T}, Y_T) =
\begin{cases}
0, 
& \text{if } X > 0 \\

k(-0.5)X, 
& \text{if } -5\% < X < 0 \\

-0.5X, 
& \text{if } X < -5\%
\end{cases}
\\[6pt]
& \text{where } X = r_{S_T} \frac{Y_T}{Y_0} - e^{r_{B_0}T}, \quad
k = \frac{X}{-5\%} \text{ (proportion of loss compensated)}
\\[12pt]

\textbf{Interest rate processes:} \quad 
& \begin{cases}
\text{US: } dr_{A_t} = \kappa_A (\theta_A - r_{A_t})\,dt + \sigma_A dW_t^A \\
\text{Japan: } dr_{B_t} = \kappa_B (\theta_B - r_{B_t})\,dt + \sigma_B dW_t^B
\end{cases} \\[12pt]

\textbf{Foreign exchange rate SDE:} \quad 
& dY_t = (r_{B_t} - r_{A_t}) Y_t\,dt + \sigma_Y Y_t\,dW_t^Y \\[10pt]

\textbf{Stock price process:} \quad 
& dS_t = r_{A_t} S_t\,dt + \sigma_S S_t\,dW_t^S

\end{aligned}
$$

$$
A = \text{USD}, \quad B = \text{JPY}, \quad Y_0 = \text{USD/JPY at } t=0, \quad Y_T = \text{USD/JPY at } t=T
$$


In [67]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

START = pd.to_datetime("2022-01-01")
END = pd.to_datetime("2026-05-31")
type(START), type(END)

(pandas.Timestamp, pandas.Timestamp)

读取美国利率

In [68]:
# 数据文件和当前 notebook 在同一个文件夹下


# Bloomberg 导出的 Excel 前 5 行是 metadata，第 6 行才是真正表头
us_bond = pd.read_excel("GB3 Govt.xlsx", sheet_name="Worksheet", skiprows=5)

# 清理列名
us_bond.columns = us_bond.columns.astype(str).str.strip()

# 处理日期
us_bond["Date"] = pd.to_datetime(us_bond["Date"])
us_bond = us_bond.loc[us_bond["Date"].between(START, END, inclusive="both")]  # 默认 inclusive="both"

# Bloomberg 利率是百分比，例如 3.60 表示 3.60%，所以要除以 100
us_bond["us"] = pd.to_numeric(us_bond["YLD_CNV_MID"], errors="coerce") / 100

# 只保留需要的列
us_bond = (
    us_bond[["Date", "us"]]
    .dropna()
    .sort_values("Date")
    .drop_duplicates(subset="Date", keep="last")
    .reset_index(drop=True)
)

print(us_bond.head())
print(us_bond.tail())

        Date       us
0 2022-01-03  0.00062
1 2022-01-04  0.00080
2 2022-01-05  0.00089
3 2022-01-06  0.00096
4 2022-01-07  0.00095
           Date       us
1144 2026-05-25  0.03667
1145 2026-05-26  0.03668
1146 2026-05-27  0.03671
1147 2026-05-28  0.03676
1148 2026-05-29  0.03675


读取日本利率

In [69]:
# Bloomberg 导出的 Excel 前 5 行是 metadata，第 6 行才是真正表头
jp_bond = pd.read_excel("GTJPY3M Govt.xlsx", sheet_name="Worksheet", skiprows=5)

# 清理列名
jp_bond.columns = jp_bond.columns.astype(str).str.strip()

# 处理日期
jp_bond["Date"] = pd.to_datetime(jp_bond["Date"])
jp_bond = jp_bond.loc[jp_bond["Date"].between(START, END, inclusive="both")]  # 默认 inclusive="both"

# Bloomberg 利率是百分比，例如 3.60 表示 3.60%，所以要除以 100
jp_bond["jp"] = pd.to_numeric(jp_bond["PX_LAST"], errors="coerce") / 100

# 只保留需要的列
jp_bond = (
    jp_bond[["Date", "jp"]]
    .dropna()
    .sort_values("Date")
    .drop_duplicates(subset="Date", keep="last")
    .reset_index(drop=True)
)

print(jp_bond.head())
print(jp_bond.tail())

        Date       jp
0 2022-01-04 -0.00150
1 2022-01-05 -0.00122
2 2022-01-06 -0.00104
3 2022-01-07 -0.00106
4 2022-01-11 -0.00104
           Date       jp
1068 2026-05-25  0.00852
1069 2026-05-26  0.00854
1070 2026-05-27  0.00857
1071 2026-05-28  0.00855
1072 2026-05-29  0.00872


读取Nasdaq指数数据

In [70]:
nasdaq=pd.read_excel("Nasdaq-100 Index (ticker NDX Index).xlsx", sheet_name="Worksheet", skiprows=6)

# 清理列名
nasdaq.columns = nasdaq.columns.astype(str).str.strip()

# 处理日期
nasdaq["Date"] = pd.to_datetime(nasdaq["Date"])
nasdaq = nasdaq.loc[nasdaq["Date"].between(START, END, inclusive="both")]  # 默认 inclusive="both"

nasdaq["close"] = pd.to_numeric(nasdaq["PX_LAST"], errors="coerce")

# 只保留需要的列
nasdaq = (
    nasdaq[["Date", "close"]]
    .dropna()
    .sort_values("Date")
    .drop_duplicates(subset="Date", keep="last")
    .reset_index(drop=True)
)

print(nasdaq.head())
print(nasdaq.tail())

        Date     close
0 2022-01-03  16501.77
1 2022-01-04  16279.73
2 2022-01-05  15771.78
3 2022-01-06  15765.36
4 2022-01-07  15592.19
           Date     close
1100 2026-05-22  29481.64
1101 2026-05-26  30001.32
1102 2026-05-27  29973.57
1103 2026-05-28  30223.89
1104 2026-05-29  30333.18


读取USD/JPY spot数据

In [71]:
FX=pd.read_excel("USDJPY Spot Exchange rate (ticker USDJPY Curncy).xlsx", sheet_name="Worksheet", skiprows=6)

# 清理列名
FX.columns = FX.columns.astype(str).str.strip()

# 处理日期
FX["Date"] = pd.to_datetime(FX["Date"])
FX = FX.loc[FX["Date"].between(START, END, inclusive="both")]  # 默认 inclusive="both"

FX["close"] = pd.to_numeric(FX["PX_LAST"], errors="coerce")

# 只保留需要的列
FX = (
    FX[["Date", "close"]]
    .dropna()
    .sort_values("Date")
    .drop_duplicates(subset="Date", keep="last")
    .reset_index(drop=True)
)

print(FX.head())
print(FX.tail())

        Date   close
0 2022-01-03  115.32
1 2022-01-04  116.16
2 2022-01-05  116.11
3 2022-01-06  115.83
4 2022-01-07  115.56
           Date   close
1145 2026-05-25  158.91
1146 2026-05-26  159.30
1147 2026-05-27  159.52
1148 2026-05-28  159.24
1149 2026-05-29  159.27


In [72]:
# Calculate daily volatility for NASDAQ and USD/JPY using log returns
price = nasdaq["close"].values
log_price = np.log(price)  # 此行没有经济意义
log_return = np.diff(log_price)  # size比原数据小1
sigma_S = np.std(log_return, ddof=1) * np.sqrt(252)   # 默认ddof=0计算总体std，这里用ddof=1计算样本std，除以n-1
print("Estimated sigma_S (NASDAQ):", sigma_S)


Estimated sigma_S (NASDAQ): 0.23391754328391953


In [73]:
# Align FX and rates by date
fx_rates = (
    FX.rename(columns={"close": "Y"})
    .merge(us_bond, on="Date", how="inner")
    .merge(jp_bond, on="Date", how="inner")
    .sort_values("Date")
    .reset_index(drop=True)
)

dt = 1 / 252

fx_rates["delta_log_Y"] = np.log(fx_rates["Y"]).diff()

# Use previous day's rates to explain next log FX move
fx_rates["drift"] = (fx_rates["jp"].shift(1) - fx_rates["us"].shift(1)) * dt

fx_rates["fx_residual"] = fx_rates["delta_log_Y"] - fx_rates["drift"]

sigma_Y = fx_rates["fx_residual"].dropna().std(ddof=1) / np.sqrt(dt)

print("Estimated sigma_Y (drift-adjusted, annualized):", sigma_Y)

Estimated sigma_Y (drift-adjusted, annualized): 0.10551463105335022


对齐美国和日本市场数据的日期

In [74]:
dt = 1 / 252

# Build aligned market dataframe
market = (
    FX.rename(columns={"close": "Y"})
    .merge(nasdaq.rename(columns={"close": "S"}), on="Date", how="inner")
    .merge(us_bond, on="Date", how="inner")
    .merge(jp_bond, on="Date", how="inner")
    .sort_values("Date")
    .dropna()
    .reset_index(drop=True)
)

# Equity volatility
market["dlog_S"] = np.log(market["S"]).diff()
sigma_S = market["dlog_S"].dropna().std(ddof=1) / np.sqrt(dt)

# FX volatility, drift-adjusted
market["dlog_Y"] = np.log(market["Y"]).diff()
market["fx_drift"] = (market["jp"].shift(1) - market["us"].shift(1)) * dt
market["fx_residual"] = market["dlog_Y"] - market["fx_drift"]

sigma_Y = market["fx_residual"].dropna().std(ddof=1) / np.sqrt(dt)

print("Estimated sigma_S (NASDAQ, annualized):", sigma_S)
print("Estimated sigma_Y (USD/JPY, drift-adjusted, annualized):", sigma_Y)

print("Aligned market data shape:", market.shape)
display(market.head())
display(market.tail())

Estimated sigma_S (NASDAQ, annualized): 0.23736214787035984
Estimated sigma_Y (USD/JPY, drift-adjusted, annualized): 0.10753698087677485
Aligned market data shape: (1033, 9)


,Date,Y,S,us,jp,dlog_S,dlog_Y,fx_drift,fx_residual
0,2022-01-04,116.16,16279.73,0.00080,-0.00150,NaN,NaN,NaN,NaN
1,2022-01-05,116.11,15771.78,0.00089,-0.00122,-0.031699,-0.000431,-0.000009,-0.000421
2,2022-01-06,115.83,15765.36,0.00096,-0.00104,-0.000407,-0.002414,-0.000008,-0.002406
3,2022-01-07,115.56,15592.19,0.00095,-0.00106,-0.011045,-0.002334,-0.000008,-0.002326
4,2022-01-11,115.30,15844.12,0.00117,-0.00104,0.016028,-0.002252,-0.000008,-0.002244


,Date,Y,S,us,jp,dlog_S,dlog_Y,fx_drift,fx_residual
1028,2026-05-22,159.18,29481.64,0.03668,0.00868,0.004227,0.001257,-0.000110,0.001368
1029,2026-05-26,159.30,30001.32,0.03668,0.00854,0.017474,0.000754,-0.000111,0.000865
1030,2026-05-27,159.52,29973.57,0.03671,0.00857,-0.000925,0.001380,-0.000112,0.001492
1031,2026-05-28,159.24,30223.89,0.03676,0.00855,0.008317,-0.001757,-0.000112,-0.001645
1032,2026-05-29,159.27,30333.18,0.03675,0.00872,0.003609,0.000188,-0.000112,0.000300


In [75]:

# us_bond = us_bond.set_index("Date")

display(us_bond, jp_bond)

# yields = pd.concat([us_bond, jpn_bond], axis=1, join="inner").dropna()
# display(yields)

,Date,us
0,2022-01-03,0.00062
1,2022-01-04,0.00080
2,2022-01-05,0.00089
3,2022-01-06,0.00096
4,2022-01-07,0.00095
...,...,...
1144,2026-05-25,0.03667
1145,2026-05-26,0.03668
1146,2026-05-27,0.03671
1147,2026-05-28,0.03676


,Date,jp
0,2022-01-04,-0.00150
1,2022-01-05,-0.00122
2,2022-01-06,-0.00104
3,2022-01-07,-0.00106
4,2022-01-11,-0.00104
...,...,...
1068,2026-05-25,0.00852
1069,2026-05-26,0.00854
1070,2026-05-27,0.00857
1071,2026-05-28,0.00855


Vasicek Model 参数计算

In [76]:
# Estimate parameters for each yield series
def get_parameters(df, col, dt=1/252):
    r = (
        pd.to_numeric(df[col], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .values
    )

    r_prev = r[:-1]
    r_curr = r[1:]
    d_r = r_curr - r_prev

    reg = LinearRegression().fit(r_prev.reshape(-1, 1), d_r)

    a = reg.intercept_
    b = reg.coef_[0]

    kappa = -b / dt
    theta = -a / b if b != 0 else np.nan

    fitted = reg.predict(r_prev.reshape(-1, 1))
    residuals = d_r - fitted
    sigma = np.std(residuals, ddof=1) / np.sqrt(dt)

    half_life = np.log(2) / kappa if kappa > 0 else np.nan
    r2 = reg.score(r_prev.reshape(-1, 1), d_r)

    result = {
        "kappa": kappa,
        "theta": theta,
        "sigma": sigma,
        "intercept": a,
        "slope": b,
        "r2": r2,
        "half_life_years": half_life,
        "n_obs": len(r)
    }

    print(f"For {col}, estimated Vasicek parameters:")
    for key, value in result.items():
        print(f"{key}: {value}")
    print()

    return result


dt = 1 / 252

us_params = get_parameters(us_bond, "us", dt=dt)
jp_params = get_parameters(jp_bond, "jp", dt=dt)

For us, estimated Vasicek parameters:
kappa: 0.9270069029400083
theta: 0.04933467285309342
sigma: 0.0052959834614753
intercept: 0.00018148246940120886
slope: -0.003678598821190509
r2: 0.02429873154812401
half_life_years: 0.7477260183949274
n_obs: 1149

For jp, estimated Vasicek parameters:
kappa: 0.01927666080307939
theta: 0.12563108432682685
sigma: 0.003591791494221303
intercept: 9.610110313060732e-06
slope: -7.64946857265055e-05
r2: 1.1071609543611416e-06
half_life_years: 35.95784496292102
n_obs: 1073



导出数据

In [77]:
import json

params = {
    "us": us_params,
    "jp": jp_params,
    "sigma_S": float(sigma_S),
    "sigma_Y": float(sigma_Y),
    "start": str(START.date()),
    "end": str(END.date())
}

with open("estimated_parameters.json", "w") as f:
    json.dump(params, f, indent=4)

print(params)

{'us': {'kappa': np.float64(0.9270069029400083), 'theta': np.float64(0.04933467285309342), 'sigma': np.float64(0.0052959834614753), 'intercept': np.float64(0.00018148246940120886), 'slope': np.float64(-0.003678598821190509), 'r2': 0.02429873154812401, 'half_life_years': np.float64(0.7477260183949274), 'n_obs': 1149}, 'jp': {'kappa': np.float64(0.01927666080307939), 'theta': np.float64(0.12563108432682685), 'sigma': np.float64(0.003591791494221303), 'intercept': np.float64(9.610110313060732e-06), 'slope': np.float64(-7.64946857265055e-05), 'r2': 1.1071609543611416e-06, 'half_life_years': np.float64(35.95784496292102), 'n_obs': 1073}, 'sigma_S': 0.23736214787035984, 'sigma_Y': 0.10753698087677485, 'start': '2022-01-01', 'end': '2026-05-31'}


robustness check

In [78]:
# Robustness windows for Japan 3M government yield Vasicek estimates
# Re-read the full Japan rate history so longer windows are not limited by START.
jp_bond_full = pd.read_excel("GTJPY3M Govt.xlsx", sheet_name="Worksheet", skiprows=5)
jp_bond_full.columns = jp_bond_full.columns.astype(str).str.strip()
jp_bond_full["Date"] = pd.to_datetime(jp_bond_full["Date"])
jp_bond_full["jp"] = pd.to_numeric(jp_bond_full["PX_LAST"], errors="coerce") / 100
jp_bond_full = (
    jp_bond_full[["Date", "jp"]]
    .dropna()
    .sort_values("Date")
    .drop_duplicates(subset="Date", keep="last")
    .reset_index(drop=True)
)


def estimate_jp_window(label, start_date, end_date=END, dt=1/252):
    sample = jp_bond_full.loc[
        jp_bond_full["Date"].between(pd.to_datetime(start_date), pd.to_datetime(end_date), inclusive="both")
    ].copy()

    r = sample["jp"].values
    r_prev = r[:-1]
    r_curr = r[1:]
    d_r = r_curr - r_prev

    reg = LinearRegression().fit(r_prev.reshape(-1, 1), d_r)
    a = reg.intercept_
    b = reg.coef_[0]

    kappa = -b / dt
    theta = -a / b if b != 0 else np.nan
    fitted = reg.predict(r_prev.reshape(-1, 1))
    residuals = d_r - fitted
    sigma = np.std(residuals, ddof=1) / np.sqrt(dt)
    half_life = np.log(2) / kappa if kappa > 0 else np.nan

    return {
        "window": label,
        "sample_start": sample["Date"].min().date(),
        "sample_end": sample["Date"].max().date(),
        "n_obs": len(r),
        "kappa": kappa,
        "theta": theta,
        "sigma": sigma,
        "half_life_years": half_life,
        "r2": reg.score(r_prev.reshape(-1, 1), d_r)
    }


windows = [
    ("2010-2026", "2010-01-01"),
    ("2016-2026", "2016-01-01"),
    ("2020-2026", "2020-01-01"),
    ("2022-2026", "2022-01-01"),
    ("2024-2026", "2024-01-01"),
]

jp_window_table = pd.DataFrame(
    [estimate_jp_window(label, start_date) for label, start_date in windows]
).set_index("window")

display(
    jp_window_table.style.format({
        "kappa": "{:.4f}",
        "theta": "{:.4%}",
        "sigma": "{:.4%}",
        "half_life_years": "{:.2f}",
        "r2": "{:.4f}",
    })
)

,sample_start,sample_end,n_obs,kappa,theta,sigma,half_life_years,r2
window,,,,,,,,
2010-2026,2010-01-01,2026-05-29,3508,0.8866,0.0563%,0.3726%,0.78,0.0011
2016-2026,2016-01-04,2026-05-29,2328,0.8256,0.0736%,0.4471%,0.84,0.0009
2020-2026,2020-01-06,2026-05-29,1552,-0.0214,-7.4013%,0.3293%,nan,0.0000
2022-2026,2022-01-04,2026-05-29,1073,0.0193,12.5631%,0.3592%,35.96,0.0000
2024-2026,2024-01-04,2026-05-29,585,0.3068,1.6363%,0.3531%,2.26,0.0002
